In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
import pyspark.sql.functions as F
import pyspark.sql.types as T
import re
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-structured-streaming-preparation").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 18:19:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill: Extended Tweet Analytics

The provided consumer already extracts and counts `@mentions`.

Your tasks:

1. **Hashtags** — also extract `#hashtags` using `re.findall(r"#[a-zA-Z0-9_]+", line)` and count them by value. Save to `out/hashtags`.

2. **URLs** — extract full URLs using `re.findall(r"https?://[^\s\"]+", line)` and count by value. Save to `out/urls`.

3. **Domains** — derive the domain from each URL by splitting on `"/"` and taking index `[2]` (e.g., `"https://spark.apache.org/releases/"` → `"spark.apache.org"`). Count domains by value and save to `out/domains`.

**Workflow:**
1. Run the **simulator cell** first.
2. Run the **consumer stub cell** — implement the three TODOs.
3. Interrupt the kernel (■) to stop the stream after a few batches.

In [2]:
TWEETS = [
    "Excited to attend #SIGMOD2025 next week! @databricks talk on streaming #bigdata",
    "Just deployed @ApacheSpark 4.0 in prod #datapipeline #streaming https://spark.apache.org/releases/",
    "@hanisaf great lecture on #java streams today map/filter/reduce clicked #uga #cs",
    "New paper on watermark strategies https://arxiv.org/abs/2501.00001 #dataengineering @databricks",
    "Watching @GoDawgs game while finishing #Spark homework #uga #football",
    "@ApacheSpark vs @hadoopconference which session? #bigdata #conference",
    "@delta_lake game-changing for streaming+batch https://delta.io/blog/2024-lakehouse/ #lakehouses",
    "Live keynote https://conf.example.org/live @ApacheKafka topic partitioning #kafka #streaming",
    "Office hours today 3pm #uga students welcome @hanisaf https://calendar.google.com/event",
    "Unified batch/stream APIs in @ApacheSpark #spark #dataengineering #streaming",
    "@ApacheKafka 4.0 queue mode is a game changer #kafka #streaming https://kafka.apache.org/",
    "Flink vs Spark https://blog.example.com/flink-vs-spark #flink #spark @ApacheKafka",
    "@hanisaf when does the midterm grade post? #uga https://canvas.uga.edu",
    "#Python taking over #dataengineering even Java shops switching @pydata https://pydata.org/",
    "Structured vs DStream: which in 2025? #spark #streaming @ApacheSpark https://spark.apache.org/streaming/",
]

for t in TWEETS:
    print(t)

Excited to attend #SIGMOD2025 next week! @databricks talk on streaming #bigdata
Just deployed @ApacheSpark 4.0 in prod #datapipeline #streaming https://spark.apache.org/releases/
@hanisaf great lecture on #java streams today map/filter/reduce clicked #uga #cs
New paper on watermark strategies https://arxiv.org/abs/2501.00001 #dataengineering @databricks
Watching @GoDawgs game while finishing #Spark homework #uga #football
@ApacheSpark vs @hadoopconference which session? #bigdata #conference
@delta_lake game-changing for streaming+batch https://delta.io/blog/2024-lakehouse/ #lakehouses
Live keynote https://conf.example.org/live @ApacheKafka topic partitioning #kafka #streaming
Office hours today 3pm #uga students welcome @hanisaf https://calendar.google.com/event
Unified batch/stream APIs in @ApacheSpark #spark #dataengineering #streaming
@ApacheKafka 4.0 queue mode is a game changer #kafka #streaming https://kafka.apache.org/
Flink vs Spark https://blog.example.com/flink-vs-spark #flin

In [3]:
# ── Tweet Simulator (provided) ────────────────────────────────────────────
# Run this cell FIRST, then run the consumer cell below.

import socket, threading, time

def run_simulator(port=7777, delay=1.5):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((STREAM_HOST, port))
    srv.listen(1)
    print(f"Simulator: waiting for Spark on port {port} ...")
    conn, addr = srv.accept()
    print(f"Simulator: connected from {addr}. Streaming ...")
    i = 0
    try:
        while True:
            conn.sendall((TWEETS[i % len(TWEETS)] + "\n").encode())
            i += 1
            time.sleep(delay)
    except (BrokenPipeError, ConnectionResetError):
        print("Simulator: Spark disconnected.")
    finally:
        conn.close(); srv.close()

sim_thread = threading.Thread(target=run_simulator, daemon=True)
sim_thread.start()
print("Simulator ready — now run the Spark consumer cell.")

Simulator ready — now run the Spark consumer cell.


Exception in thread Thread-6 (run_simulator):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_73959/1178008716.py", line 9, in run_simulator
OSError: [Errno 99] Cannot assign requested address


# Your Task

In [4]:
# ── Stub — extend the Spark consumer ─────────────────────────────────────
ssc = StreamingContext(sc, 10)   # 10-second batch interval

lines = ssc.socketTextStream(STREAM_HOST, 7777)

watch_keywords = ["@hanisaf", "#uga", "#spark", "#bigdata"]

# Provided: mention extraction and counting
mentions      = lines.flatMap(lambda line: re.findall(r"@[a-zA-Z0-9_]+", line))
mention_counts = mentions.countByValue()
mention_counts.pprint()

# TODO 1: Extract #hashtags and count by value, then pprint() them.
#         Save hashtag counts to "out/hashtags"
hashtags = None   # replace with flatMap + findAll

# TODO 2: Extract URLs (pattern: r"https?://[^\s\"]+") and count by value.
#         Save URL counts to "out/urls"
urls = None       # replace with flatMap + findAll

# TODO 3: Extract the domain from each URL.
#         A URL like "https://spark.apache.org/releases/" has domain "spark.apache.org"
#         Hint: split the URL on "/" and take index [2]
domains = None    # replace with transform on urls DStream

mention_counts.saveAsTextFiles("out/mentions", "txt")

print("Streaming started — interrupt the kernel to stop.")
ssc.start()
ssc.awaitTermination(30)
ssc.stop(stopSparkContext=False)

/usr/local/lib/python3.12/site-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


Streaming started — interrupt the kernel to stop.


-------------------------------------------
Time: 2026-06-28 18:19:20
-------------------------------------------



-------------------------------------------
Time: 2026-06-28 18:19:30
-------------------------------------------



-------------------------------------------
Time: 2026-06-28 18:19:40
-------------------------------------------
('@databricks', 2)
('@delta_lake', 1)
('@hanisaf', 3)
('@pydata', 1)
('@GoDawgs', 1)
('@hadoopconference', 1)
('@ApacheKafka', 3)
('@ApacheSpark', 4)



26/06/28 18:19:46 ERROR ReceiverTracker: Deregistered receiver for stream 0: Stopped by driver


In [6]:
spark.stop()